# DSA 4060 – Week 3 Practical Lab: Building a Simple Content-Based Recommender

**Student ID:** 669648

**Objective:** Build a simple content-based movie recommender that uses structured genre features. The workflow is Movie Features → User Preferences → Recommendation Scores → Top Recommendations. I build the item-feature matrix, create a user profile from ratings, score and rank movies, remove the ones already watched, and return Top-N recommendations. I also test a second user, add my own movie and user, and discuss the limitations.

## 1. Import libraries

In [1]:
import pandas as pd

## 2. Create the dataset

In [2]:
movies = pd.DataFrame({
    "movie_id": [1,2,3,4,5,6,7,8,9,10],
    "title": [
        "Galaxy Mission","Love in Nairobi","Final Battle","Family Weekend",
        "Future World","Broken Hearts","City Detectives","Space Warriors",
        "Wedding Weekend","Hidden Truth"
    ],
    "Action": [1,0,1,0,1,0,1,1,0,0],
    "Comedy": [0,0,0,1,0,0,0,0,1,0],
    "Drama": [0,1,0,0,0,1,1,0,0,1],
    "Romance": [0,1,0,0,0,1,0,0,1,0],
    "SciFi": [1,0,0,0,1,0,0,1,0,0]
})
movies.to_csv("movies.csv", index=False)
movies

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1
1,2,Love in Nairobi,0,0,1,1,0
2,3,Final Battle,1,0,0,0,0
3,4,Family Weekend,0,1,0,0,0
4,5,Future World,1,0,0,0,1
5,6,Broken Hearts,0,0,1,1,0
6,7,City Detectives,1,0,1,0,0
7,8,Space Warriors,1,0,0,0,1
8,9,Wedding Weekend,0,1,0,1,0
9,10,Hidden Truth,0,0,1,0,0


Feature encoding: 1 means the genre is present and 0 means it is absent. For example, Galaxy Mission has Action = 1 and SciFi = 1, so its item profile is [1, 0, 0, 0, 1].

## 3. Explore the dataset

In [3]:
movies.head()

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1
1,2,Love in Nairobi,0,0,1,1,0
2,3,Final Battle,1,0,0,0,0
3,4,Family Weekend,0,1,0,0,0
4,5,Future World,1,0,0,0,1


In [4]:
movies.shape

(10, 7)

In [5]:
movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   movie_id  10 non-null     int64
 1   title     10 non-null     str  
 2   Action    10 non-null     int64
 3   Comedy    10 non-null     int64
 4   Drama     10 non-null     int64
 5   Romance   10 non-null     int64
 6   SciFi     10 non-null     int64
dtypes: int64(6), str(1)
memory usage: 692.0 bytes


**Answers**

1. There are **10 movies** in the dataset (the shape is 10 rows × 7 columns).
2. There are **5 genre features**: Action, Comedy, Drama, Romance and SciFi.
3. **One row represents one movie**, with its ID, its title and five binary flags showing which genres it belongs to.

## 4. Identify structured features

In [6]:
features = ["Action", "Comedy", "Drama", "Romance", "SciFi"]
movie_features = movies[features]
movie_features

,Action,Comedy,Drama,Romance,SciFi
0,1,0,0,0,1
1,0,0,1,1,0
2,1,0,0,0,0
3,0,1,0,0,0
4,1,0,0,0,1
5,0,0,1,1,0
6,1,0,1,0,0
7,1,0,0,0,1
8,0,1,0,1,0
9,0,0,1,0,0


This is the item-feature matrix. Each row is a movie, each column is a genre, and the recommender only sees these 0/1 values.

## 5. Create item profiles

In [7]:
movies.loc[
    movies["title"] == "Galaxy Mission"
]

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1


In the Galaxy Mission profile, the 1s under Action and SciFi mean the movie belongs to those two genres. The 0s under Comedy, Drama and Romance mean it has none of those, so its item profile is [1, 0, 0, 0, 1].

## 6. Create user ratings

In [8]:
ratings = pd.DataFrame({
    "movie_id": [1, 2, 3, 5],
    "rating": [5, 2, 4, 5]
})
ratings

,movie_id,rating
0,1,5
1,2,2
2,3,4
3,5,5


In [9]:
user_movies = ratings.merge(
    movies,
    on="movie_id"
)
user_movies

,movie_id,rating,title,Action,Comedy,Drama,Romance,SciFi
0,1,5,Galaxy Mission,1,0,0,0,1
1,2,2,Love in Nairobi,0,0,1,1,0
2,3,4,Final Battle,1,0,0,0,0
3,5,5,Future World,1,0,0,0,1


In [10]:
user_movies[
    ["title", "rating", "Action", "Comedy", "Drama", "Romance", "SciFi"]
]

,title,rating,Action,Comedy,Drama,Romance,SciFi
0,Galaxy Mission,5,1,0,0,0,1
1,Love in Nairobi,2,0,0,1,1,0
2,Final Battle,4,1,0,0,0,0
3,Future World,5,1,0,0,0,1


**Observation before any calculation:** User A rated Galaxy Mission (5), Future World (5) and Final Battle (4) highly. These are all Action movies, and two of them are also SciFi. The only low rating (2) is for Love in Nairobi, which is Drama and Romance. So the user seems to prefer Action and SciFi and does not seem to like Drama or Romance much.

## 7. Build the user profile

In [11]:
rated_features = user_movies[features]
user_ratings = user_movies["rating"]
rated_features

,Action,Comedy,Drama,Romance,SciFi
0,1,0,0,0,1
1,0,0,1,1,0
2,1,0,0,0,0
3,1,0,0,0,1


In [12]:
user_ratings

0    5
1    2
2    4
3    5
Name: rating, dtype: int64

In [13]:
weighted_features = rated_features.mul(
    user_ratings,
    axis=0)
weighted_features

,Action,Comedy,Drama,Romance,SciFi
0,5,0,0,0,5
1,0,0,2,2,0
2,4,0,0,0,0
3,5,0,0,0,5


Multiplying each movie's features by the rating means a highly rated movie contributes more to the profile than a lowly rated one.

In [14]:
user_profile = weighted_features.sum()
user_profile

Action     14
Comedy      0
Drama       2
Romance     2
SciFi      10
dtype: int64

In [15]:
user_profile.sort_values(
    ascending=False
)

Action     14
SciFi      10
Drama       2
Romance     2
Comedy      0
dtype: int64

**Strongest genres:** The two strongest genres for User A are **Action (14)** and **SciFi (10)**. Action gets 5 + 4 + 5 from Galaxy Mission, Final Battle and Future World, and SciFi gets 5 + 5 from Galaxy Mission and Future World. Drama and Romance only get 2 each (from the 2-star Love in Nairobi) and Comedy gets 0.

In [16]:
user_profile_normalized = (
    user_profile / user_profile.sum()
)
user_profile_normalized

Action     0.500000
Comedy     0.000000
Drama      0.071429
Romance    0.071429
SciFi      0.357143
dtype: float64

After normalizing, the profile is in proportions that add up to 1. Action is 50% of this user's preference, SciFi about 36%, Drama and Romance about 7% each, and Comedy 0%.

## 8. Calculate recommendation scores

In [17]:
movies["recommendation_score"] = (
    movie_features
    .mul(user_profile_normalized)
    .sum(axis=1)
)
movies[["title", "recommendation_score"]]

,title,recommendation_score
0,Galaxy Mission,0.857143
1,Love in Nairobi,0.142857
2,Final Battle,0.500000
3,Family Weekend,0.000000
4,Future World,0.857143
5,Broken Hearts,0.142857
6,City Detectives,0.571429
7,Space Warriors,0.857143
8,Wedding Weekend,0.071429
9,Hidden Truth,0.071429


In [18]:
ranked_movies = movies.sort_values(
    "recommendation_score",
    ascending=False
)
ranked_movies[["title", "recommendation_score"]]

,title,recommendation_score
0,Galaxy Mission,0.857143
4,Future World,0.857143
7,Space Warriors,0.857143
6,City Detectives,0.571429
2,Final Battle,0.500000
1,Love in Nairobi,0.142857
5,Broken Hearts,0.142857
8,Wedding Weekend,0.071429
9,Hidden Truth,0.071429
3,Family Weekend,0.000000


A higher score means the movie contains more of the features this user prefers. Space Warriors scores highest because it has both Action and SciFi.

## 9. Remove watched movies

In [19]:
watched_movies = ratings["movie_id"]
recommendations = ranked_movies[
    ~ranked_movies["movie_id"].isin(watched_movies)
]
recommendations[["title", "recommendation_score"]]

,title,recommendation_score
7,Space Warriors,0.857143
6,City Detectives,0.571429
5,Broken Hearts,0.142857
8,Wedding Weekend,0.071429
9,Hidden Truth,0.071429
3,Family Weekend,0.000000


## 10. Generate Top-N recommendations

In [20]:
top_3 = recommendations[
    ["title", "recommendation_score"]
].head(3)
top_3

,title,recommendation_score
7,Space Warriors,0.857143
6,City Detectives,0.571429
5,Broken Hearts,0.142857


**Why each movie was recommended**

- **Space Warriors (0.857):** It is both Action and SciFi, which are the user's two strongest preferences (0.50 + 0.36), so it gets the highest score.
- **City Detectives (0.571):** It is an Action movie, and Action is the user's strongest genre. The Drama tag adds only a little.
- **Broken Hearts (0.143):** It does not match Action or SciFi at all. It only gets a small score from Drama and Romance, which the user rated low, and it is third simply because the other unwatched movies score even lower.

### Reusable function

In [21]:
def recommend_movies(movies, ratings, features, n=3):
    user_movies = ratings.merge(movies, on="movie_id")
    rated_features = user_movies[features]
    weighted_features = rated_features.mul(
        user_movies["rating"],
        axis=0
    )
    user_profile = weighted_features.sum()
    user_profile = user_profile / user_profile.sum()
    scores = (
        movies[features]
        .mul(user_profile)
        .sum(axis=1)
    )
    results = movies.copy()
    results["recommendation_score"] = scores
    watched = ratings["movie_id"]
    results = results[
        ~results["movie_id"].isin(watched)
    ]
    return results.sort_values(
        "recommendation_score", ascending=False
    ).head(n)

In [22]:
recommend_movies(
    movies,
    ratings,
    features
)

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
7,8,Space Warriors,1,0,0,0,1,0.857143
6,7,City Detectives,1,0,1,0,0,0.571429
5,6,Broken Hearts,0,0,1,1,0,0.142857


## 11. Test a second user

In [23]:
user_b_ratings = pd.DataFrame({
    "movie_id": [2, 4, 6, 9],
    "rating": [5, 4, 5, 4]
})
recommend_movies(
    movies,
    user_b_ratings,
    features
)

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
9,10,Hidden Truth,0,0,1,0,0,0.3125
6,7,City Detectives,1,0,1,0,0,0.3125
2,3,Final Battle,1,0,0,0,0,0.0000


**User A vs User B:** No, they did not get the same recommendations. User A prefers Action and SciFi, so the top results are Space Warriors and City Detectives. User B rated Love in Nairobi, Broken Hearts (both Drama/Romance), Family Weekend and Wedding Weekend (Comedy) highly, so the profile is built from Romance (14), Drama (10) and Comedy (8) with 0 for Action and SciFi. Because of that, the only unwatched movies with a non-zero score are the Drama ones, City Detectives and Hidden Truth, which tie at 0.3125. The rest have a score of 0, so the third place is just whichever zero-score movie is sorted first. This shows the recommender really does depend on the individual user's profile, but it also shows that with so few features there are lots of ties.

## 12. Thinking exercise: User C

User C has only watched Galaxy Mission and rated it 5 stars.

4. **What will the model learn about User C?** It will learn that User C likes Action and SciFi, because those are the only two features of the only movie they rated.
5. **Which genres dominate?** Action and SciFi, each with a profile score of 5. After normalizing, each is exactly 0.5 and everything else is 0.
6. **What could go wrong?** The recommendations will only be Action/SciFi movies. Anything with Comedy, Drama or Romance gets a score of 0, even if User C would actually enjoy it. The model also cannot tell whether User C likes Action, SciFi or just that one movie.
7. **Is one rating enough?** No. One rating is far too little to understand a user's preferences. It is the cold-start problem for a new user: the profile is based on a single data point, so it is extremely narrow and could be quite wrong.

## 13. Add my unique movie

In [24]:
new_movie = {
    "movie_id": 11,
    "title": "Safari Heist",
    "Action": 1,
    "Comedy": 1,
    "Drama": 0,
    "Romance": 0,
    "SciFi": 0
}

movies = pd.concat(
    [movies, pd.DataFrame([new_movie])],
    ignore_index=True
)
movies = movies.drop(columns="recommendation_score", errors="ignore")
movies.to_csv("movies.csv", index=False)
movies

,movie_id,title,Action,Comedy,Drama,Romance,SciFi
0,1,Galaxy Mission,1,0,0,0,1
1,2,Love in Nairobi,0,0,1,1,0
2,3,Final Battle,1,0,0,0,0
3,4,Family Weekend,0,1,0,0,0
4,5,Future World,1,0,0,0,1
5,6,Broken Hearts,0,0,1,1,0
6,7,City Detectives,1,0,1,0,0
7,8,Space Warriors,1,0,0,0,1
8,9,Wedding Weekend,0,1,0,1,0
9,10,Hidden Truth,0,0,1,0,0


I added **Safari Heist**, an Action + Comedy movie. None of the original ten movies has this exact combination of genres, so it is a unique item profile: [1, 1, 0, 0, 0]. I also removed the leftover `recommendation_score` column from Step 8 and saved the updated 11-movie dataset to `movies.csv`.

## 14. Create my own user

In [25]:
my_user = pd.DataFrame({
    "movie_id": [7, 10, 11, 6, 5],
    "rating": [5, 4, 5, 2, 3]
})
my_user.merge(movies[["movie_id", "title"]], on="movie_id")

,movie_id,rating,title
0,7,5,City Detectives
1,10,4,Hidden Truth
2,11,5,Safari Heist
3,6,2,Broken Hearts
4,5,3,Future World


In [26]:
recommend_movies(
    movies,
    my_user,
    features,
    n=5
)

,movie_id,title,Action,Comedy,Drama,Romance,SciFi,recommendation_score
0,1,Galaxy Mission,1,0,0,0,1,0.470588
7,8,Space Warriors,1,0,0,0,1,0.470588
1,2,Love in Nairobi,0,0,1,1,0,0.382353
2,3,Final Battle,1,0,0,0,0,0.382353
8,9,Wedding Weekend,0,1,0,1,0,0.205882


## 15. Interpretation

8. **Two strongest preferences:** Action (13) and Drama (11). I rated City Detectives (Action + Drama) 5, Hidden Truth (Drama) 4 and Safari Heist (Action + Comedy) 5, so those two genres have the highest weighted totals. Comedy comes next with 5.
9. **Three highest scores:** Galaxy Mission (0.471), Space Warriors (0.471) and Love in Nairobi (0.382), with Final Battle also at 0.382.
10. **Why the top movie ranked first:** Galaxy Mission and Space Warriors are both Action + SciFi. Action is my strongest genre (13/34 = 0.38) and SciFi adds 3/34 = 0.09, so they score 0.47. They tie because they have identical item profiles, so the model has no way to prefer one over the other.
11. **Surprises:** Yes. I expected movies mixing Action and Drama to come out on top, but the two Action + SciFi movies won. Love in Nairobi also ranked third even though it is a Drama/Romance movie, because Drama is my second strongest genre. Both come from the fact that the model adds up genre scores and does not care about the combination.
12. **What the model ignores:** Actors, directors, release year, language, plot descriptions, how popular a movie is, and what other users think. It also ignores the order and time of ratings, and it treats all genres as equally important parts of a movie.

## 16. Limitations

1. **Only five features.** A movie can only be described by five genre flags, so very different movies end up with identical profiles (Galaxy Mission and Space Warriors) and tie.
2. **Lots of ignored metadata.** Actors, directors, release year, language and descriptions are not used, even though a user might like a particular director or hate a certain language.
3. **Over-specialisation.** The model only recommends movies similar to what the user already liked, so a user who likes Action will keep getting Action movies and never discover anything new.
4. **Cold start.** A new user like User C has little or no history, so the profile is very narrow and unreliable.
5. **Zero scores for unseen genres.** If a user has never rated a genre, every movie in that genre scores 0 however good it is.
6. **Quality depends on the metadata.** If the genre labels are wrong or too broad, the recommendations will be wrong too.

## 17. Conclusion

In this lab I built a content-based recommender with pandas using only structured genre features. It builds a user profile by weighting the item features by the user's ratings, normalizes the profile, scores every movie, removes the ones already watched and returns the Top-N. It gave different recommendations for User A and User B, which shows it personalises, and the unique movie and my own user showed how it works on new data. The tests also showed its limits: ties from identical profiles, over-specialisation and cold start. These are the reasons Week 4 moves on to richer text features such as TF-IDF and cosine similarity.

## Final Question

*If two users have watched exactly the same movies but gave them different ratings, should this recommender produce the same recommendations for both users?*

Not necessarily. The user profile is built from the ratings (rating × genre features), so different ratings give different profiles even when the watched movies are the same. The candidate set (the unwatched movies) is identical for both users, but the scores and ranking can change. The test below uses the same four movies as User A with different ratings.

In [27]:
same_movies_a = pd.DataFrame({"movie_id": [1, 2, 3, 5], "rating": [5, 2, 4, 5]})
same_movies_b = pd.DataFrame({"movie_id": [1, 2, 3, 5], "rating": [2, 5, 2, 2]})

print("User with ratings 5, 2, 4, 5")
display(recommend_movies(movies, same_movies_a, features, n=3)[["title", "recommendation_score"]])
print("User with ratings 2, 5, 2, 2")
display(recommend_movies(movies, same_movies_b, features, n=3)[["title", "recommendation_score"]])

User with ratings 5, 2, 4, 5


,title,recommendation_score
7,Space Warriors,0.857143
6,City Detectives,0.571429
10,Safari Heist,0.500000


User with ratings 2, 5, 2, 2


,title,recommendation_score
6,City Detectives,0.55
5,Broken Hearts,0.50
7,Space Warriors,0.50


Both users watched Galaxy Mission, Love in Nairobi, Final Battle and Future World. The first user liked the Action/SciFi movies, so Space Warriors and City Detectives come out on top. The second user gave Love in Nairobi 5 and the others 2, so the Drama and Romance weights go up and the top recommendation becomes City Detectives, followed by Broken Hearts, with Space Warriors dropping down to a tie. So the recommender uses the ratings and not just which movies were watched, which is how it should behave.